In [1]:
!pip -q install antlr4-python3-runtime==4.13.2
!curl -sSLo antlr-4.13.2-complete.jar https://www.antlr.org/download/antlr-4.13.2-complete.jar

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\USER\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!java -version
!ls -lh antlr-4.13.2-complete.jar

java version "21.0.7" 2025-04-15 LTS
Java(TM) SE Runtime Environment (build 21.0.7+8-LTS-245)
Java HotSpot(TM) 64-Bit Server VM (build 21.0.7+8-LTS-245, mixed mode, sharing)
"ls" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [3]:
%%writefile StoreLang.g4

grammar StoreLang;


// ==== PARSER

programa
    : construccion* EOF
    ;

construccion
    : producto
    | proveedor
    | compra
    | venta
    | reporte
    ;


// ----  PRODUCTO

producto
    : PRODUCTO ID LBRACE nombre precio categoria RBRACE
    ;

nombre
    : NOMBRE ASSIGN CADENA SEMI
    ;

precio
    : PRECIO ASSIGN DECIMAL SEMI
    ;

categoria
    : CATEGORIA ASSIGN CADENA SEMI
    ;


// ----  PROVEEDOR

proveedor
    : PROVEEDOR ID LBRACE nombreProveedor contacto RBRACE
    ;

nombreProveedor
    : NOMBRE ASSIGN CADENA SEMI
    ;

contacto
    : CONTACTO ASSIGN CADENA SEMI
    ;


// ----  COMPRA

compra
    : COMPRA ID LBRACE proveedorRef productoRef cantidad costo RBRACE
    ;

proveedorRef
    : PROVEEDOR ASSIGN ID SEMI
    ;

productoRef
    : PRODUCTO ASSIGN ID SEMI
    ;

cantidad
    : CANTIDAD ASSIGN ENTERO SEMI
    ;

costo
    : COSTO ASSIGN DECIMAL SEMI
    ;


// ----  VENTA

venta
    : VENTA ID LBRACE listaItems RBRACE
    ;

listaItems
    : itemVenta+
    ;

itemVenta
    : PRODUCTO ASSIGN ID SEMI
      CANTIDAD ASSIGN ENTERO SEMI
    ;


// ----  REPORTE

reporte
    : REPORTE tipoReporte filtro? SEMI
    ;

tipoReporte
    : STOCK
    | VENTAS
    | PRODUCTOS
    ;

filtro
    : CATEGORIA ASSIGN CADENA
    | PRODUCTO ASSIGN ID
    ;


// ==== LEXER

// ----  Palabras reservadas
PRODUCTO    : 'producto';
PROVEEDOR   : 'proveedor';
COMPRA      : 'compra';
VENTA       : 'venta';
REPORTE     : 'reporte';

NOMBRE      : 'nombre';
PRECIO      : 'precio';
CATEGORIA   : 'categoria';
CONTACTO    : 'contacto';

CANTIDAD    : 'cantidad';
COSTO       : 'costo';

STOCK       : 'stock';
VENTAS      : 'ventas';
PRODUCTOS   : 'productos';

// ----  Símbolos
LBRACE      : '{';
RBRACE      : '}';
ASSIGN      : '=';
SEMI        : ';';

// ----  Valores
DECIMAL
    : [0-9]+ '.' [0-9]+
    ;

ENTERO
    : [0-9]+
    ;

CADENA
    : '"' ~[\r\n"]* '"'
    ;

ID
    : [a-zA-Z_][a-zA-Z0-9_]*
    ;

// ----  Espacios y comentarios
WS
    : [ \t\r\n]+ -> skip
    ;

COMMENT
    : '//' ~[\r\n]* -> skip
    ;

Writing StoreLang.g4


In [4]:
# generar el lexer y parser
!java -jar antlr-4.13.2-complete.jar -Dlanguage=Python3 -visitor StoreLang.g4

In [5]:
# funcion para analizar el codigo

from antlr4 import *
from StoreLangLexer import StoreLangLexer
from StoreLangParser import StoreLangParser
from antlr4.error.ErrorListener import ErrorListener


class SyntaxErrorListener(ErrorListener):

    def __init__(self):
        super().__init__()
        self.errors = []

    def syntaxError(self, recognizer, offendingSymbol, line, column, msg, e):
        self.errors.append(
            f"Línea {line}, columna {column}: {msg}"
        )


def analizar_codigo(codigo, mostrar_arbol=False):

    # Input
    input_stream = InputStream(codigo)

    # Lexer
    lexer = StoreLangLexer(input_stream)

    lexer_errors = SyntaxErrorListener()
    lexer.removeErrorListeners()
    lexer.addErrorListener(lexer_errors)

    token_stream = CommonTokenStream(lexer)

    # Parser
    parser = StoreLangParser(token_stream)

    parser_errors = SyntaxErrorListener()
    parser.removeErrorListeners()
    parser.addErrorListener(parser_errors)

    tree = parser.programa()

    print("========== RESULTADO ==========")

    if not lexer_errors.errors and not parser_errors.errors:
        print("Análisis sintáctico correcto.")
    else:

        if lexer_errors.errors:
            print("Errores léxicos:")
            for error in lexer_errors.errors:
                print("  -", error)

        if parser_errors.errors:
            print("Errores sintacticos:")
            for error in parser_errors.errors:
                print("  -", error)

    if mostrar_arbol:
        print("\n========== ARBOL ==========")
        print(tree.toStringTree(recog=parser))

    return tree, lexer_errors.errors, parser_errors.errors

ModuleNotFoundError: No module named 'antlr4'

In [ ]:
# ---- Pruebas
# 1er Programa válido
codigo_valido = """
producto P001 {
    nombre = "Arroz";
    precio = 4.50;
    categoria = "Abarrotes";
}

proveedor PR01 {
    nombre = "Distribuidora Lima";
    contacto = "987654321";
}

compra C001 {
    proveedor = PR01;
    producto = P001;
    cantidad = 50;
    costo = 3.20;
}

venta V001 {
    producto = P001;
    cantidad = 3;
}

reporte stock;
"""

tree, lex_errors, syn_errors = analizar_codigo(
    codigo_valido,
    mostrar_arbol=True
)

========== RESULTADO ==========
Análisis sintáctico correcto.

========== ARBOL ==========
(programa (construccion (producto producto P001 { (nombre nombre = "Arroz" ;) (precio precio = 4.50 ;) (categoria categoria = "Abarrotes" ;) })) (construccion (proveedor proveedor PR01 { (nombreProveedor nombre = "Distribuidora Lima" ;) (contacto contacto = "987654321" ;) })) (construccion (compra compra C001 { (proveedorRef proveedor = PR01 ;) (productoRef producto = P001 ;) (cantidad cantidad = 50 ;) (costo costo = 3.20 ;) })) (construccion (venta venta V001 { (listaItems (itemVenta producto = P001 ; cantidad = 3 ;)) })) (construccion (reporte reporte (tipoReporte stock) ;)) <EOF>)


In [ ]:
# verificamos que listaItems permite varios productos
codigo_venta_multiple = """
producto P001 {
    nombre = "Arroz";
    precio = 4.50;
    categoria = "Abarrotes";
}

venta V002 {
    producto = P001;
    cantidad = 2;
    producto = P002;
    cantidad = 1;
}

reporte ventas;
"""

tree, lex_errors, syn_errors = analizar_codigo(
    codigo_venta_multiple,
    mostrar_arbol=False
)

========== RESULTADO ==========
Análisis sintáctico correcto.


Nótese que el parser sí acepta P002 aunque todavía no esté declarado.

Es correcto, porque estamos a nivel sintáctico, y el parser revisa que la estructura sea correcta.

Si existe o no P002 es trabajo del analizador semántico

In [ ]:
# Pruebas de errores sintacticos
codigo_error_1 = """
producto P001 {
    nombre = "Arroz"
    precio = 4.50;
    categoria = "Abarrotes";
}
"""

tree, lex_errors, syn_errors = analizar_codigo(
    codigo_error_1,
    mostrar_arbol=False
)

========== RESULTADO ==========
Errores sintacticos:
  - Línea 4, columna 4: missing ';' at 'precio'


In [ ]:
codigo_error_2 = """
compra C001 {
    proveedor = PR01;
    producto = P001;
    cantidad = "hola"
    costo = 3.20;
}
"""

tree, lex_errors, syn_errors = analizar_codigo(
    codigo_error_2,
    mostrar_arbol=False
)

========== RESULTADO ==========
Errores sintacticos:
  - Línea 5, columna 15: mismatched input '"hola"' expecting ENTERO


In [ ]:
# estructura incorrecta
codigo_error_3 = """
producto P001
    nombre = "Arroz";
    precio = 4.50;
    categoria = "Abarrotes";
}
"""

tree, lex_errors, syn_errors = analizar_codigo(
    codigo_error_3,
    mostrar_arbol=False
)

========== RESULTADO ==========
Errores sintacticos:
  - Línea 3, columna 4: missing '{' at 'nombre'


In [ ]:
# Prueba del lexer
codigo_lexico = '''
producto P001 {
    nombre = "Arroz";
    precio = 4.50;
    categoria = "Abarrotes";
}
'''

input_stream = InputStream(codigo_lexico)
lexer = StoreLangLexer(input_stream)

tokens = lexer.getAllTokens()

for token in tokens:
    print(
        f"{token.line}:{token.column} "
        f"{token.text:<20} "
        f"{lexer.symbolicNames[token.type]}"
    )

2:0 producto             PRODUCTO
2:9 P001                 ID
2:14 {                    LBRACE
3:4 nombre               NOMBRE
3:11 =                    ASSIGN
3:13 "Arroz"              CADENA
3:20 ;                    SEMI
4:4 precio               PRECIO
4:11 =                    ASSIGN
4:13 4.50                 DECIMAL
4:17 ;                    SEMI
5:4 categoria            CATEGORIA
5:14 =                    ASSIGN
5:16 "Abarrotes"          CADENA
5:27 ;                    SEMI
6:0 }                    RBRACE


In [ ]:
# Analizador Semántico
from StoreLangVisitor import StoreLangVisitor

In [ ]:
class SemanticAnalyzer(StoreLangVisitor):

    def __init__(self):
        super().__init__()

        self.productos = {}
        self.proveedores = {}
        self.operaciones = {}

        self.errors = []

    def error(self, mensaje):
        self.errors.append(mensaje)


    # ---- PRIMERA PASADA
    # Registrar declaraciones

    def analizar_declaraciones(self, tree):

        for construccion in tree.construccion():

            if construccion.producto():

                ctx = construccion.producto()
                identificador = ctx.ID().getText()

                if identificador in self.productos:
                    self.error(
                        f"Producto duplicado: '{identificador}'"
                    )
                else:
                    self.productos[identificador] = ctx

            elif construccion.proveedor():

                ctx = construccion.proveedor()
                identificador = ctx.ID().getText()

                if identificador in self.proveedores:
                    self.error(
                        f"Proveedor duplicado: '{identificador}'"
                    )
                else:
                    self.proveedores[identificador] = ctx


    # ---- SEGUNDA PASADA
    # Validar referencias y reglas


    def analizar_referencias(self, tree):

        for construccion in tree.construccion():


            # --- COMPRA
            if construccion.compra():

                ctx = construccion.compra()

                operacion_id = ctx.ID().getText()

                self.validar_operacion(
                    operacion_id,
                    "compra"
                )

                proveedor_id = (
                    ctx.proveedorRef()
                    .ID()
                    .getText()
                )

                producto_id = (
                    ctx.productoRef()
                    .ID()
                    .getText()
                )

                if proveedor_id not in self.proveedores:
                    self.error(
                        f"Proveedor inexistente en compra "
                        f"'{operacion_id}': '{proveedor_id}'"
                    )

                if producto_id not in self.productos:
                    self.error(
                        f"Producto inexistente en compra "
                        f"'{operacion_id}': '{producto_id}'"
                    )

                cantidad = int(
                    ctx.cantidad()
                    .ENTERO()
                    .getText()
                )

                if cantidad <= 0:
                    self.error(
                        f"La cantidad de la compra "
                        f"'{operacion_id}' debe ser mayor que 0"
                    )


            # ---- VENTA
            elif construccion.venta():

                ctx = construccion.venta()

                operacion_id = ctx.ID().getText()

                self.validar_operacion(
                    operacion_id,
                    "venta"
                )

                for item in ctx.listaItems().itemVenta():

                    producto_id = (
                        item.ID()
                        .getText()
                    )

                    cantidad = int(
                        item.ENTERO()
                        .getText()
                    )

                    if producto_id not in self.productos:
                        self.error(
                            f"Producto inexistente en venta "
                            f"'{operacion_id}': '{producto_id}'"
                        )

                    if cantidad <= 0:
                        self.error(
                            f"La cantidad de la venta "
                            f"'{operacion_id}' debe ser mayor que 0"
                        )

    def validar_operacion(self, identificador, tipo):

        if identificador in self.operaciones:

            self.error(
                f"ID de operación duplicado: "
                f"'{identificador}'"
            )

        else:
            self.operaciones[identificador] = tipo


    # ---- MÉTODO PRINCIPAL

    def analizar(self, tree):

        self.analizar_declaraciones(tree)
        self.analizar_referencias(tree)

        return self.errors

In [ ]:
# Programa correcto:
codigo_semanticamente_correcto = """
producto P001 {
    nombre = "Arroz";
    precio = 4.50;
    categoria = "Abarrotes";
}

proveedor PR01 {
    nombre = "Distribuidora Lima";
    contacto = "987654321";
}

compra C001 {
    proveedor = PR01;
    producto = P001;
    cantidad = 50;
    costo = 3.20;
}

venta V001 {
    producto = P001;
    cantidad = 3;
}

reporte stock;
"""

In [ ]:
# analisis lexico y sintactico
tree, lex_errors, syn_errors = analizar_codigo(
    codigo_semanticamente_correcto
)

========== RESULTADO ==========
Análisis sintáctico correcto.


In [ ]:
# analisis semantico
semantic = SemanticAnalyzer()

semantic_errors = semantic.analizar(tree)

print("========== ANÁLISIS SEMÁNTICO ==========")

if not semantic_errors:
    print("No se encontraron errores semánticos.")
else:
    for error in semantic_errors:
        print("Error: ", error)

========== ANÁLISIS SEMÁNTICO ==========
No se encontraron errores semánticos.


In [ ]:
# Error semántico: Producto inexistente
codigo_error_semantico_1 = """
producto P001 {
    nombre = "Arroz";
    precio = 4.50;
    categoria = "Abarrotes";
}

proveedor PR01 {
    nombre = "Distribuidora Lima";
    contacto = "987654321";
}

venta V001 {
    producto = P999;
    cantidad = 3;
}
"""

In [ ]:
tree, lex_errors, syn_errors = analizar_codigo(
    codigo_error_semantico_1
)

semantic = SemanticAnalyzer()
semantic_errors = semantic.analizar(tree)

print("========== ANÁLISIS SEMÁNTICO ==========")

for error in semantic_errors:
    print("Error: ", error)

========== RESULTADO ==========
Análisis sintáctico correcto.
========== ANÁLISIS SEMÁNTICO ==========
Error:  Producto inexistente en venta 'V001': 'P999'


In [ ]:
# Error semántico: cantidad inválida
codigo_error_semantico_2 = """
producto P001 {
    nombre = "Arroz";
    precio = 4.50;
    categoria = "Abarrotes";
}

venta V001 {
    producto = P001;
    cantidad = 0;
}
"""

In [ ]:
tree, lex_errors, syn_errors = analizar_codigo(
    codigo_error_semantico_2
)

semantic = SemanticAnalyzer()
semantic_errors = semantic.analizar(tree)

print("========== ANÁLISIS SEMÁNTICO ==========")

for error in semantic_errors:
    print("Error: ", error)

========== RESULTADO ==========
Análisis sintáctico correcto.
========== ANÁLISIS SEMÁNTICO ==========
Error:  La cantidad de la venta 'V001' debe ser mayor que 0


In [ ]:
# Error semántico: ID duplicado
codigo_error_semantico_3 = """
producto P001 {
    nombre = "Arroz";
    precio = 4.50;
    categoria = "Abarrotes";
}

producto P001 {
    nombre = "Azucar";
    precio = 3.80;
    categoria = "Abarrotes";
}
"""

In [ ]:
tree, lex_errors, syn_errors = analizar_codigo(
    codigo_error_semantico_3
)

semantic = SemanticAnalyzer()
semantic_errors = semantic.analizar(tree)

print("========== ANÁLISIS SEMÁNTICO ==========")

for error in semantic_errors:
    print("Error: ", error)

========== RESULTADO ==========
Análisis sintáctico correcto.
========== ANÁLISIS SEMÁNTICO ==========
Error:  Producto duplicado: 'P001'


In [ ]:
# FUNCIÓN PARA REALIZAR UNA PRUEBA COMPLETA
def ejecutar_storelang(codigo):

    print("\n========== CÓDIGO FUENTE ==========")

    print(codigo)


    # ---- FASE 1: LEXER + PARSER

    tree, lex_errors, syn_errors = analizar_codigo(codigo)

    if lex_errors or syn_errors:
        print("\nError: El programa no supera el análisis sintáctico.")
        return


    # ---- FASE 2: SEMANTICA

    semantic = SemanticAnalyzer()
    semantic_errors = semantic.analizar(tree)

    print("\n========== ANÁLISIS SEMÁNTICO ==========")

    if semantic_errors:

        for error in semantic_errors:
            print("Error: ", error)

        print(
            "\nError:  El programa contiene errores semánticos."
        )

    else:

        print(
            "Correcto: El programa supera el análisis semántico."
        )

In [ ]:
ejecutar_storelang(codigo_valido)


========== CÓDIGO FUENTE ==========

producto P001 {
    nombre = "Arroz";
    precio = 4.50;
    categoria = "Abarrotes";
}

proveedor PR01 {
    nombre = "Distribuidora Lima";
    contacto = "987654321";
}

compra C001 {
    proveedor = PR01;
    producto = P001;
    cantidad = 50;
    costo = 3.20;
}

venta V001 {
    producto = P001;
    cantidad = 3;
}

reporte stock;

========== RESULTADO ==========
Análisis sintáctico correcto.

========== ANÁLISIS SEMÁNTICO ==========
Correcto: El programa supera el análisis semántico.


In [ ]:
ejecutar_storelang(codigo_error_semantico_1)


========== CÓDIGO FUENTE ==========

producto P001 {
    nombre = "Arroz";
    precio = 4.50;
    categoria = "Abarrotes";
}

proveedor PR01 {
    nombre = "Distribuidora Lima";
    contacto = "987654321";
}

venta V001 {
    producto = P999;
    cantidad = 3;
}

========== RESULTADO ==========
Análisis sintáctico correcto.

========== ANÁLISIS SEMÁNTICO ==========
Error:  Producto inexistente en venta 'V001': 'P999'

Error:  El programa contiene errores semánticos.
